In [ ]:
# =========================
# LIMPEZA COMPLETA
# =========================

import pandas as pd
import numpy as np
import os

print("Iniciando processamento...")

# =========================
# 1. CARREGAR DADOS
# =========================

caminho = 'data/dados_brutos/'
arquivos = os.listdir(caminho)

lista_dfs = []

for arquivo in arquivos:
    if arquivo.endswith('.csv'):
        try:
            df_temp = pd.read_csv(caminho + arquivo, sep=',', encoding='utf-8')
        except:
            df_temp = pd.read_csv(caminho + arquivo, sep=';', encoding='latin1')
        
        df_temp['arquivo_origem'] = arquivo
        lista_dfs.append(df_temp)

df = pd.concat(lista_dfs, ignore_index=True)

print("Total de registros carregados:", df.shape)

# =========================
# 2. PADRONIZAR COLUNAS
# =========================

df.columns = df.columns.str.strip()
df.columns = df.columns.str.lower()

print("Colunas:", df.columns)

# =========================
# 3. LIMPEZA
# =========================

# remover linhas vazias
df = df.dropna(how='all')

# remover duplicados
df = df.drop_duplicates()

print("Após limpeza:", df.shape)

# =========================
# 4. PADRONIZAR DADOS
# =========================

if 'estado' in df.columns:
    df['estado'] = df['estado'].astype(str).str.upper().str.strip()

if 'sexo' in df.columns:
    df['sexo'] = df['sexo'].astype(str).str.upper().str.strip()

# =========================
# 5. CONVERTER TIPOS
# =========================

if 'ano' in df.columns:
    df['ano'] = pd.to_numeric(df['ano'], errors='coerce')

if 'idade' in df.columns:
    df['idade'] = pd.to_numeric(df['idade'], errors='coerce')

# =========================
# 6. REMOVER NULOS IMPORTANTES
# =========================

colunas_importantes = ['estado', 'ano']

for col in colunas_importantes:
    if col in df.columns:
        df = df[df[col].notnull()]

# =========================
# 7. CRIAR VARIÁVEIS
# =========================

# coluna de contagem
df['casos'] = 1

# faixa etária
if 'idade' in df.columns:
    df['faixa_etaria'] = pd.cut(
        df['idade'],
        bins=[0, 18, 30, 45, 60, 120],
        labels=['0-18', '19-30', '31-45', '46-60', '60+']
    )

# =========================
# 8. AGRUPAMENTO (ANÁLISE)
# =========================

if 'estado' in df.columns and 'ano' in df.columns:
    base_agrupada = df.groupby(['estado', 'ano'])['casos'].sum().reset_index()

    print("\nResumo por ano:")
    print(base_agrupada.groupby('ano')['casos'].sum())

# =========================
# 9. SALVAR DADOS
# =========================

saida = 'data/dados_tratados/'
os.makedirs(saida, exist_ok=True)

# base limpa
df.to_csv(saida + 'base_limpa.csv', index=False)

# base agregada
if 'base_agrupada' in locals():
    base_agrupada.to_csv(saida + 'base_agrupada.csv', index=False)

print("\nProcessamento concluído com sucesso!")